# Gemma 4 31B on vXPU, from a notebook with no GPU

This notebook runs the examples from the Hugging Face
[Gemma 4 model docs](https://huggingface.co/docs/transformers/en/model_doc/gemma4)
against `google/gemma-4-31B-it` **without a local GPU, CUDA, or any
downloaded weights**. The code is the docs' code with one import changed:

```python
from vxpu import AutoModelForCausalLM   # instead of: from transformers import AutoModelForCausalLM
```

Behind that line, `from_pretrained` instantiates the model on torch's
`meta` device, exports a weightless artifact (manifest + two
`torch.export` graphs, ~75 MB), ships it to the **vXPU router** in the
cluster, and waits for an executor. `generate()` sends token ids and
gets token ids back; the executor, a PyTorch worker pod, runs the same
prefill/decode graphs and the same sampling rules `transformers` would
run locally. The tokenizer, processor, chat template, tools and
streamers are the ordinary `transformers` objects.

**Where it runs is the router's decision.** Gemma 4 31B is 62.6 GB in
bf16, more than a 24 GB NVIDIA L4 holds, so on an L4 cluster the router
sizes a CPU executor (~75 GiB RAM) instead; a model that fits (Gemma 4
E4B, 16 GB) goes to the GPU. Same notebook either way, slower on CPU.

## Prerequisites

- `kubectl` pointing at a GKE cluster, the vXPU images, and the router
  running (`vxpu up`); see [README.md](README.md).
- This notebook's environment: `uv pip install -e python/ jupyter pillow torchvision`
  (CPU-only torch is fine; pillow/torchvision are for the docs'
  `AutoProcessor`, which also wraps the image front-end).

In [1]:
import os, subprocess, time
import torch

print("CUDA available on this machine:", torch.cuda.is_available())
MODEL_ID = "google/gemma-4-31B-it"

CUDA available on this machine: False


## 1. Load the model

`from_pretrained` exports on first use (about a minute for the 31B,
cached under `~/.cache/vxpu`), then ships the artifact and waits for
the executor. A cold load of 62.6 GB takes 10–15 minutes; a warm
executor answers in seconds. Arguments like `device_map` are accepted
and ignored: placement is the executor's.

In [2]:
from vxpu import AutoModelForCausalLM

started = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto",
                                             artifact_dir=os.environ.get("VXPU_ARTIFACT_DIR"))
print(f"\n{model}\nready in {time.time() - started:.0f}s")

vxpu: ignoring from_pretrained arguments ['device_map']; placement, dtype and attention are the executor's decision


shipping artifact /tmp/vxpu-artifacts/gemma-4-31b-it (74 MB of graphs; weights stay remote)


executor placed and artifact delivered (515s); rehydrating weights...
  loading... (515s)


  loading... (530s)


  loading... (546s)


...

  loading... (681s)


  loading... (696s)


  loading... (711s)


model ready in 727s



VxpuModelForCausalLM('google/gemma-4-31B-it', router='127.0.0.1:57298', session='092821c865aeaaa1481ead907c24da18:s0')
ready in 729s


In [3]:
# Where did the router put it?
print(subprocess.run(
    ["kubectl", "get", "pods", "-l", "app=vxpu-executor", "-o",
     "custom-columns=POD:.metadata.name,NODE:.spec.nodeName,CPU:.spec.containers[0].resources.requests.cpu,"
     "MEMORY:.spec.containers[0].resources.requests.memory,GPU:.spec.containers[0].resources.limits.nvidia\\.com/gpu"],
    capture_output=True, text=True).stdout)

POD                                              NODE                                                  CPU   MEMORY        GPU
vxpu-executor-092821c865aeaaa1481ead907c24da18   gke-ai-us-central1-b-c3dstandard60lss-cad71bbc-wv98   30    79256664634   <none>



## 2. The Hugging Face examples

### Causal language modeling

The docs' `generate`/`batch_decode` lines, unchanged. Two things about
the prompt: Gemma 4's tokenizer does not prepend `<bos>` outside the chat
template, and an instruction-tuned Gemma given a bare prompt without its
turn template tends to loop (we observed "condiment condiment ..."), so
the prompt goes through `apply_chat_template` like any Gemma chat
prompt. `max_new_tokens` replaces `max_length=30`, which would leave only
eleven tokens for the reply after the templated prompt. Gemma 4's `generation_config` samples
(`do_sample=True`, temperature 1.0, top-k 64, top-p 0.95), and so does
`generate()` here; pass `do_sample=False` for greedy, or `seed=` for a
repeatable sample.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

prompt = "What is your favorite condiment?"
inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}],
                                       add_generation_prompt=True, return_tensors="pt", return_dict=True)

# Generate
generate_ids = model.generate(inputs.input_ids, max_new_tokens=48)
print(tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0])
print("\n", model.last)

user
What is your favorite condiment?
model
thought
As an AI, I don’t have taste buds, but if I had to choose a favorite based on "personality," I’d go with **Sriracha**.

It’s versatile, has a bit of a kick,

 GenerateStats(finish_reason='length', prompt_tokens=19, prefilled_tokens=19, generated=48, prefill_ms=1567, ms_per_token=1412.8)


### Function calling

The docs' code, unchanged: the processor renders the chat template with
the tool, `generate` runs, and the decoded output keeps the special
tokens (`skip_special_tokens=False`) so the tool call is visible. The
model stops after `<tool_call|>` because `<|tool_response>` is one of
its end-of-generation ids.

In [5]:
from transformers import AutoProcessor

WEATHER_TOOL = {
    "type": "function",
    "function": {
        "name": "get_n_day_weather_forecast",
        "description": "Get an N-day weather forecast",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA",
                },
                "format": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "The temperature unit to use",
                },
                "num_days": {
                    "type": "integer",
                    "description": "The number of days to forecast",
                },
            },
            "required": ["location", "format", "num_days"],
        },
    },
}

messages = [
    {
        "role": "user",
        "content": "What's the weather like the next 3 days in San Francisco, CA (using F)?",
    },
]

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    padding_side="left"
)

text = processor.apply_chat_template(
    messages,
    tools=[WEATHER_TOOL],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(outputs[0][input_len:], skip_special_tokens=False))

<|tool_call>call:get_n_day_weather_forecast{format:<|"|>fahrenheit<|"|>,location:<|"|>San Francisco, CA<|"|>,num_days:3}<tool_call|><|tool_response>


Completing the tool loop is ordinary `transformers` usage: parse the
call, append it as a structured `tool_calls` message plus a `tool`
response, re-render, generate. The executor keeps the session's KV
cache and prefills only from the first token that differs, so the
second turn does not pay for the tool declarations again
(`model.last.prefilled_tokens` vs `prompt_tokens`). After a tool
response Gemma 4 opens its reply with an empty `<|channel>thought`
block; decoded with `skip_special_tokens=True` that leaves a bare
"thought" line, as it would with `transformers`.

In [6]:
import json, re

def parse_gemma_tool_call(text):
    # Parse the first <|tool_call>call:name{k:<|"|>v<|"|>,k2:3}<tool_call|> block.
    m = re.search(r"<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>", text, re.S)
    if not m:
        return None
    name, body = m.group(1), m.group(2)
    args = {}
    for key, quoted, bare in re.findall(r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,]+))', body, re.S):
        if quoted:
            args[key] = quoted
        elif re.fullmatch(r"-?\d+(\.\d+)?|true|false|null", bare.strip()):
            args[key] = json.loads(bare.strip())
        else:
            args[key] = bare.strip()
    return {"name": name, "arguments": args}

call = parse_gemma_tool_call(processor.decode(outputs[0][input_len:], skip_special_tokens=False))
if call is None:
    raise ValueError("no <|tool_call>...<tool_call|> block in the reply")
print("parsed:", call)

messages.append({"role": "assistant", "tool_calls": [{"id": "call_1", "type": "function", "function": call}]})
messages.append({
    "role": "tool",
    "tool_call_id": "call_1",
    "name": call["name"],
    "content": json.dumps({
        "location": "San Francisco, CA",
        "format": "fahrenheit",
        "forecast": [
            {"day": 1, "high": 68, "low": 54, "conditions": "sunny"},
            {"day": 2, "high": 65, "low": 53, "conditions": "fog in the morning"},
            {"day": 3, "high": 62, "low": 52, "conditions": "light rain"},
        ],
    }),
})

text = processor.apply_chat_template(messages, tools=[WEATHER_TOOL], tokenize=False, add_generation_prompt=True)
inputs = processor(text=text, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=160)
print(processor.decode(outputs[0][input_len:], skip_special_tokens=True))
print("\n", model.last)

parsed: {'name': 'get_n_day_weather_forecast', 'arguments': {'format': 'fahrenheit', 'location': 'San Francisco, CA', 'num_days': 3}}


thought
The weather forecast for San Francisco, CA over the next 3 days is:

*   **Day 1:** High of 68°F, low of 54°F, and sunny.
*   **Day 2:** High of 65°F, low of 53°F, with fog in the morning.
*   **Day 3:** High of 62°F, low of 52°F, with light rain.

 GenerateStats(finish_reason='eos', prompt_tokens=320, prefilled_tokens=157, generated=106, prefill_ms=5652, ms_per_token=1483.5)


### Multi-turn chat with streaming

Not in the docs, but it is the everyday `transformers` pattern: keep a
`messages` list, re-render, `generate` with a `TextStreamer`. The KV
cache makes the follow-up turn prefill only what is new (with Gemma 4's
template that includes the previous reply, because the generation
prompt's empty `thought` channel is not re-rendered).

In [7]:
from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
chat = [{"role": "user", "content": "Give me three facts about Kubernetes, one sentence each."}]
ids = tokenizer.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"]
out = model.generate(ids, max_new_tokens=120, streamer=streamer)
chat.append({"role": "assistant", "content": tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)})
print("\n", model.last)

Kubernetes 

is 

an 

open-source 

orchestration 

...

for 

cloud 

infrastructure 

management.



 GenerateStats(finish_reason='eos', prompt_tokens=24, prefilled_tokens=22, generated=72, prefill_ms=1879, ms_per_token=1470.6)


In [8]:
chat.append({"role": "user", "content": "Which of those three matters most for running AI workloads? Answer briefly."})
ids = tokenizer.apply_chat_template(chat, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"]
out = model.generate(ids, max_new_tokens=96, streamer=streamer)
print("\n", model.last)

The 

second 

fact 

matters 

most, 

...

required 

by 

AI 

workloads.



 GenerateStats(finish_reason='eos', prompt_tokens=119, prefilled_tokens=99, generated=29, prefill_ms=3487, ms_per_token=1468.8)


### Configuration

Pure architecture code from the docs; it needs no GPU. We build it on
the `meta` device so it needs no RAM either, which is exactly how the
export behind `from_pretrained` works. (Keyword arguments: transformers
5 no longer accepts the sub-configs positionally.)

In [9]:
from transformers import (
    Gemma4AudioConfig,
    Gemma4Config,
    Gemma4ForConditionalGeneration,
    Gemma4TextConfig,
    Gemma4VisionConfig,
)

# Initializing a Gemma 4 Audio config.
audio_config = Gemma4AudioConfig()

# Initializing a Gemma 4 Text config.
text_config = Gemma4TextConfig()

# Initializing a Gemma 4 vision config.
vision_config = Gemma4VisionConfig()

# Initializing a Gemma 4 config similar to google/gemma-4-e2b-it
configuration = Gemma4Config(text_config=text_config, vision_config=vision_config, audio_config=audio_config)

# Initializing a model from the google/gemma-4-e2b-it configuration (weightless: meta device)
with torch.device("meta"):
    meta_model = Gemma4ForConditionalGeneration(configuration)

# Accessing the model configuration
configuration = meta_model.config
print(type(meta_model).__name__, "with", sum(p.numel() for p in meta_model.parameters()) / 1e9, "B parameters (on", next(meta_model.parameters()).device, ")")

Gemma4ForConditionalGeneration with 5.554675488 B parameters (on meta )


### Image and audio examples

The docs also show image-text-to-text (pipeline and `AutoModel`) and
audio transcription. They are not run here: the exported artifact
captures the **text** decoder path (`input_ids` → logits), so the vision
tower is not part of the graphs the executor runs, and audio is native
only on the E2B/E4B sizes (the 31B has `audio_config: null`).
Multimodal export is a vXPU roadmap item.

## 3. Clean up

The executor keeps the model resident for an hour of idleness and the
pods stay up; remove them when done:

```sh
vxpu down
kubectl delete pods -l app=vxpu-executor
```

In [10]:
model.close()